# Passo 2b – Revisão dos boatos e conjunto eleitoral final

Este notebook:

1. Lê as decisões da revisão feita com apoio de modelo de linguagem (`revisao_claude_lote_XX.csv`).
2. Lê a sua conferência (`conferencia_lote_01_para_Davi.xlsx`) e mede a **concordância** entre as duas anotações.
3. Monta o **conjunto eleitoral final** (`fpc_eleitoral_final.csv`).

**Pré-requisito:** o `02_limpeza_e_extracao.ipynb` foi executado. Os arquivos de decisão e a sua planilha preenchida devem estar na pasta `TCC/resultados/revisao` do Drive.

**Regra de resolução:** nos itens que você conferiu, a **sua decisão prevalece**. Nos demais, vale a decisão da revisão assistida.

In [ ]:
import os, glob
import numpy as np
import pandas as pd
from sklearn.metrics import cohen_kappa_score, confusion_matrix

try:
    from google.colab import drive
    drive.mount('/content/drive')
    PASTA_TCC = '/content/drive/MyDrive/TCC'
except ImportError:
    PASTA_TCC = os.path.abspath('TCC')

REV = os.path.join(PASTA_TCC, 'resultados', 'revisao')
PROC = os.path.join(PASTA_TCC, 'dados', 'processados')
RES = os.path.join(PASTA_TCC, 'resultados')
ARQUIVO_DAVI = os.path.join(REV, 'conferencia_lote_01_para_Davi.xlsx')
VALIDAS = ['ok', 'corrigir', 'descartar']

cand = pd.read_csv(os.path.join(REV, 'revisao_completa.csv'))[['id', 'lote', 'agencia', 'titulo', 'link', 'regra', 'trecho']]
print('Candidatos a boato:', len(cand))

In [ ]:
# Se as decisões não estiverem na pasta, baixa do repositório público
if not glob.glob(os.path.join(REV, 'revisao_claude_lote_*.csv')):
    import subprocess, shutil
    subprocess.run(['git','clone','--depth','1','https://github.com/DaviWillian03/tcc-daviwillian','/tmp/tcc_repo'], check=True)
    for f in glob.glob('/tmp/tcc_repo/revisao/revisao_claude_lote_*.csv'):
        shutil.copy(f, REV)
print(sorted(os.path.basename(f) for f in glob.glob(os.path.join(REV, 'revisao_claude_lote_*.csv'))))

## 1. Decisões da revisão assistida

In [ ]:
arqs = sorted(glob.glob(os.path.join(REV, 'revisao_claude_lote_*.csv')))
print('Arquivos de decisão encontrados:', [os.path.basename(a) for a in arqs])
claude = pd.concat([pd.read_csv(a) for a in arqs], ignore_index=True)
claude['decisao'] = claude['decisao'].astype(str).str.strip().str.lower()

assert claude['id'].is_unique, 'há ids repetidos nas decisões'
assert set(claude['decisao']) <= set(VALIDAS), f"decisão inválida: {set(claude['decisao']) - set(VALIDAS)}"
assert set(claude['id']) <= set(cand['id']), 'há ids que não existem nos candidatos'

print(f"Revisados: {len(claude)} de {len(cand)} candidatos ({100*len(claude)/len(cand):.0f}%)")
t = cand.merge(claude, on='id')
tabela = t.groupby('agencia')['decisao'].value_counts().unstack(fill_value=0).reindex(columns=VALIDAS, fill_value=0)
tabela['total'] = tabela.sum(axis=1)
tabela['% aproveitado'] = (100 * (tabela['ok'] + tabela['corrigir']) / tabela['total']).round(0)
tabela

## 2. Concordância com a sua conferência

Medidas:
- **Concordância simples:** % de itens em que as duas decisões foram iguais.
- **Kappa de Cohen:** desconta a concordância que aconteceria por acaso. Escala de Landis e Koch (1977): 0,61 a 0,80 é concordância substancial; acima de 0,80, quase perfeita.
- Calculado de duas formas: com as 3 categorias e com 2 (aproveitado = ok ou corrigir, contra descartado). A de 2 categorias é a que importa para o conjunto final.

In [ ]:
if not os.path.exists(ARQUIVO_DAVI):
    print('Planilha de conferência ainda não encontrada. Pulando esta seção.')
    davi = None
else:
    davi = pd.read_excel(ARQUIVO_DAVI, skiprows=1)
    davi.columns = ['id', 'agencia', 'titulo', 'link', 'contexto', 'trecho', 'trecho_corrigido_davi', 'decisao_davi']
    davi['decisao_davi'] = davi['decisao_davi'].astype(str).str.strip().str.lower().replace({'nan': ''})
    vazias = (davi['decisao_davi'] == '').sum()
    invalidas = davi[~davi['decisao_davi'].isin(VALIDAS + [''])]
    print(f'Itens na planilha: {len(davi)} | sem decisão: {vazias} | decisão inválida: {len(invalidas)}')
    if len(invalidas):
        print(invalidas[['id', 'decisao_davi']].to_string())
    davi = davi[davi['decisao_davi'].isin(VALIDAS)].copy()

    comp = davi.merge(claude[['id', 'decisao']], on='id', how='inner')
    assert len(comp) == len(davi), 'há itens da planilha sem decisão da revisão assistida'

    concord = (comp['decisao'] == comp['decisao_davi']).mean() * 100
    kappa3 = cohen_kappa_score(comp['decisao'], comp['decisao_davi'], labels=VALIDAS)
    aprov = lambda s: np.where(s == 'descartar', 'descartar', 'aproveitar')
    a, b = aprov(comp['decisao']), aprov(comp['decisao_davi'])
    concord2 = (a == b).mean() * 100
    kappa2 = cohen_kappa_score(a, b)

    print(f'\nItens comparados: {len(comp)}')
    print(f'3 categorias: concordância {concord:.1f}% | kappa {kappa3:.2f}')
    print(f'2 categorias (aproveitar x descartar): concordância {concord2:.1f}% | kappa {kappa2:.2f}')
    print('\nMatriz (linhas = revisão assistida, colunas = você):')
    print(pd.DataFrame(confusion_matrix(comp['decisao'], comp['decisao_davi'], labels=VALIDAS),
                       index=VALIDAS, columns=VALIDAS).to_string())

    resumo = pd.DataFrame([{'itens': len(comp), 'concordancia_3cat_%': round(concord, 1), 'kappa_3cat': round(kappa3, 3),
                            'concordancia_2cat_%': round(concord2, 1), 'kappa_2cat': round(kappa2, 3)}])
    resumo.to_csv(os.path.join(RES, 'concordancia_revisao.csv'), index=False)

    desacordos = comp[comp['decisao'] != comp['decisao_davi']][['id', 'agencia', 'titulo', 'decisao', 'decisao_davi', 'trecho']]
    print(f'\nDesacordos ({len(desacordos)}): vale ler um por um e entender o padrão.')
    desacordos.to_csv(os.path.join(RES, 'desacordos_revisao.csv'), index=False)
    desacordos[['id', 'agencia', 'decisao', 'decisao_davi']]

## 3. Conjunto eleitoral final

- Itens `ok`: entram com o trecho extraído.
- Itens `corrigir`: entram com o trecho corrigido (se vier vazio, o item é descartado e contado).
- Itens `descartar`: ficam de fora.
- Onde você conferiu, a sua decisão e o seu trecho corrigido prevalecem.

In [ ]:
final = cand.merge(claude[['id', 'decisao', 'trecho_corrigido']], on='id')
final['fonte_decisao'] = 'revisão assistida'

if davi is not None and len(davi):
    d = davi.set_index('id')
    idx = final['id'].isin(d.index)
    final.loc[idx, 'decisao'] = final.loc[idx, 'id'].map(d['decisao_davi'])
    final.loc[idx, 'trecho_corrigido'] = final.loc[idx, 'id'].map(d['trecho_corrigido_davi'])
    final.loc[idx, 'fonte_decisao'] = 'conferência do autor'

final['trecho_corrigido'] = final['trecho_corrigido'].fillna('').astype(str).str.strip()
final['texto'] = np.where((final['decisao'] == 'corrigir') & (final['trecho_corrigido'] != ''),
                          final['trecho_corrigido'], final['trecho'])

sem_correcao = (final['decisao'] == 'corrigir') & (final['trecho_corrigido'] == '')
print('Itens "corrigir" sem trecho corrigido (descartados):', int(sem_correcao.sum()))
entra = final['decisao'].isin(['ok', 'corrigir']) & ~sem_correcao
eleitoral = final[entra].copy()
eleitoral['rotulo'] = 1
eleitoral['base'] = 'FactPolCheckBr'
eleitoral = eleitoral[['id', 'base', 'agencia', 'titulo', 'link', 'texto', 'rotulo', 'decisao', 'fonte_decisao']]
eleitoral.to_csv(os.path.join(PROC, 'fpc_eleitoral_final.csv'), index=False)

print(f'\nConjunto eleitoral final: {len(eleitoral)} boatos (de {len(final)} revisados)')
print(pd.crosstab(eleitoral['agencia'], eleitoral['decisao'], margins=True).to_string())
print('\nDecisão tomada por:', eleitoral['fonte_decisao'].value_counts().to_dict())
print('Palavras por boato (mediana):', int(eleitoral['texto'].str.split().str.len().median()))